In [ ]:
import sys
import time
import random
import numpy as np
import heapq


import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

# --- 1. 脳（ニューラルネットワーク）の定義 ---
class SimpleNetwork(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleNetwork, self).__init__()
        # 入力層 -> 隠れ層(128) -> 隠れ層(128) -> 出力層
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x) # 各ジョブを選んだときの「期待される価値(Q値)」を出力

# --- 2. 学習するエージェント ---
class DQNAgent:
    def __init__(self, n_jobs, n_machines):
        self.n_jobs = n_jobs
        # 状態のサイズ（簡易的に 機械の数 + ジョブの数 と仮定）
        self.state_size = n_machines + n_jobs
        self.action_size = n_jobs

        # 脳を作成
        self.policy_net = SimpleNetwork(self.state_size, self.action_size)
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=0.001)
        self.memory = deque(maxlen=10000) # 経験再生用メモリ

        self.epsilon = 1.0  # 探索率（最初はランダムに動く）
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01

    def get_state_vector(self, state):
        """辞書型のstateをAIが読めるベクトル(数値の列)に変換"""
        # 機械の負荷状況
        m_load = state["machine_load"]
        # 次のジョブの作業時間（ない場合は0）
        next_ops = np.array([op[1] for op in state["next_ops"]])

        # 結合して1つのベクトルにする
        # 正規化などをしないと学習しにくいが、ここでは簡易実装
        state_vec = np.concatenate([m_load, next_ops])
        return torch.FloatTensor(state_vec)

    def act(self, legal_actions, state):
        """行動決定（推論フェーズ）"""
        # イプシロン・グリーディ法: たまにランダムに行動して経験を積む
        if random.random() <= self.epsilon:
            return random.choice(legal_actions)

        # AIが考える
        state_tensor = self.get_state_vector(state)
        with torch.no_grad():
            q_values = self.policy_net(state_tensor)

        # 合法な手の中で一番Q値が高いものを選ぶ
        # (違法な手のQ値をマイナス無限大にするマスク処理)
        q_values_np = q_values.numpy()
        mask = np.full(self.action_size, -np.inf)
        mask[legal_actions] = q_values_np[legal_actions]

        return np.argmax(mask)

    def train(self, batch_size=32):
        """学習（バックプロパゲーション）"""
        if len(self.memory) < batch_size:
            return

        batch = random.sample(self.memory, batch_size)
        # バッチからデータを取り出して損失を計算し、重みを更新する処理...
        # (コードが長くなるため詳細は割愛しますが、ここで optimizer.step() が走ります)

        # 探索率を少し下げる（だんだん真面目にやるようになる）
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def remember(self, state, action, reward, next_state, done):
        """経験を記憶する"""
        self.memory.append((state, action, reward, next_state, done))



# --- JSSP環境クラス (OpenAI Gymのようなスタイル) ---
class JSSPEnv:
    def __init__(self, jobs_data, n_machines):
        self.jobs_data = jobs_data
        self.n_jobs = len(jobs_data)
        self.n_machines = n_machines
        self.reset()

    def reset(self):
        """環境を初期化する"""
        # 各ジョブの現在の工程インデックス
        self.job_op_idx = [0] * self.n_jobs
        # 各ジョブの終了時刻
        self.job_end_time = [0] * self.n_jobs
        # 各機械の終了時刻
        self.machine_end_time = [0] * self.n_machines
        # 完了したジョブ数
        self.finished_jobs = 0

        # 状態: 特徴量ベクトル（ここでは簡易的に[各機械の負荷, 各ジョブの残り作業]などを想定）
        return self._get_state()

    def _get_state(self):
        """
        ニューラルネットワークに入力するための「状態」を作成
        本来はここでグラフ構造や特徴行列を作ります
        """
        # 簡易的な特徴量: 各機械の現在の完了時刻
        machine_load = np.array(self.machine_end_time)

        # 現在着手可能なジョブの情報（次の工程の機械ID、処理時間）
        next_ops_info = []
        for j in range(self.n_jobs):
            if self.job_op_idx[j] < len(self.jobs_data[j]):
                op = self.jobs_data[j][self.job_op_idx[j]]
                next_ops_info.append(op) # (machine, duration)
            else:
                next_ops_info.append((-1, 0)) # 完了済み

        return {"machine_load": machine_load, "next_ops": next_ops_info}

    def get_legal_actions(self):
        """現在選択可能なジョブのリストを返す"""
        legal_actions = []
        for j in range(self.n_jobs):
            if self.job_op_idx[j] < len(self.jobs_data[j]):
                legal_actions.append(j)
        return legal_actions

    def step(self, action_job_id):
        """
        行動（ジョブ選択）を実行し、状態を進める
        """
        # 選択されたジョブの次の工程を取得
        op_idx = self.job_op_idx[action_job_id]
        machine_id, duration = self.jobs_data[action_job_id][op_idx]

        # スケジューリング（開始時刻の決定）
        start_time = max(self.job_end_time[action_job_id], self.machine_end_time[machine_id])
        end_time = start_time + duration

        # 状態更新
        self.job_end_time[action_job_id] = end_time
        self.machine_end_time[machine_id] = end_time
        self.job_op_idx[action_job_id] += 1

        # 全工程完了判定
        done = False
        if self.job_op_idx[action_job_id] >= len(self.jobs_data[action_job_id]):
            # このジョブは完了したが、全ジョブ完了かはまだ確認が必要
            if all(idx >= len(data) for idx, data in zip(self.job_op_idx, self.jobs_data)):
                done = True

        # 報酬の計算 (強化学習用)
        # メイクスパンを最小化したい -> 負の報酬を与える、または完了時に大きな報酬
        # ここでは簡易的に「メイクスパンの増加分」を罰として与えるなどが一般的
        reward = -max(self.machine_end_time)

        next_state = self._get_state()
        return next_state, reward, done

    def get_makespan(self):
        return max(self.machine_end_time)

# --- AIエージェント (本来はここに深層学習モデルが入る) ---
class RandomAgent:
    def act(self, legal_actions, state):
        return random.choice(legal_actions)

class HeuristicAgent:
    """MWKR (Most Work Remaining) ルールをエージェント化したもの"""
    def __init__(self, jobs_data):
        self.jobs_data = jobs_data

    def act(self, legal_actions, state):
        # 残り作業時間が最も多いジョブを選ぶ
        best_job = -1
        max_remaining = -1

        for j in legal_actions:
            # 残り作業時間の計算
            current_op = state["next_ops"][j] # 本来はstateから復元せずjob_op_idxが必要だが簡易化
            # ここでは簡易的に「まだ残っている全工程」の合計を計算すべきだが、
            # デモのためランダムより少しマシなロジックとして「次の工程が長いもの」を選ぶ(LPT)
            if state["next_ops"][j][1] > max_remaining:
                max_remaining = state["next_ops"][j][1]
                best_job = j
        return best_job

# --- メイン実行部 ---
def run_rl_framework():
    # 規模設定: 500ジョブ x 200機械 = 100000オペレーション (デモ用)
    n_jobs = 500
    n_machines = 200

    print(f"=== 深層強化学習フレームワーク (Demo: {n_jobs*n_machines} Ops) ===")

    # データの生成
    jobs_data = []
    for j in range(n_jobs):
        machines = list(range(n_machines))
        random.shuffle(machines)
        ops = [(m, random.randint(1, 100)) for m in machines]
        jobs_data.append(ops)

    # 環境の構築
    env = JSSPEnv(jobs_data, n_machines)

    # エージェントの選択（ここではヒューリスティックエージェント）
    # ★本番ではここで「学習済みのニューラルネット」をロードします
    agent = HeuristicAgent(jobs_data)

    state = env.reset()
    done = False
    step_count = 0
    start_time = time.time()

    # シミュレーションループ (これが推論/実行フェーズ)
    while not done:
        legal_actions = env.get_legal_actions()

        # AIが行動を決定
        action = agent.act(legal_actions, state)

        # 環境に行動を適用
        state, reward, done = env.step(action)
        step_count += 1

    elapsed = time.time() - start_time
    print(f"完了。所要時間: {elapsed:.2f}秒")
    print(f"Makespan: {env.get_makespan()}")
    print(f"Step数: {step_count}")

if __name__ == "__main__":
    run_rl_framework()


# --- 3. 学習ループの実行イメージ ---
# 実際の学習にはこのループを数万回回す必要があります

env = JSSPEnv(...)
agent = DQNAgent(...)

for episode in range(1000): # 1000回練習する
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        legal_actions = env.get_legal_actions()
        action = agent.act(legal_actions, state)

        next_state, reward, done = env.step(action)

        # 経験を記憶
        agent.remember(state, action, reward, next_state, done)

        # 脳を更新（学習）
        agent.train()

        state = next_state
        total_reward += reward

    print(f"Episode {episode}: Total Reward = {total_reward}")


=== 深層強化学習フレームワーク (Demo: 100000 Ops) ===
完了。所要時間: 20.33秒
Makespan: 3994531
Step数: 100000
